# Research script: are daily market probabilities informative?

This notebook is the research layer for the filtered daily Kalshi snapshot. It keeps only the analysis needed for the first empirical question: whether daily market probabilities are closer to the final outcomes than simple reference forecasts.

The unit of analysis is a resolved binary market. We use `price_mean` as the market probability and evaluate it 1, 3, and 7 calendar days before close. No predictive model is trained here.

## Load the filtered snapshot

The files below contain the filtered market metadata and the corresponding daily candles. The small inventory table confirms what this analysis is actually using.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

DATA_DIR = Path("../data/demo_filtered")
if not DATA_DIR.exists():
    DATA_DIR = Path("data/demo_filtered")
if not DATA_DIR.exists():
    DATA_DIR = Path("daily_export/data/demo_filtered")
METADATA_PATH = DATA_DIR / "main_market_metadata.csv"
CANDLES_PATH = DATA_DIR / "main_daily_candles.csv.gz"

metadata_text = [
    "market_id", "market_question", "event_question",
    "series_title", "series_category", "market_result",
    "download_status",
]
metadata = pd.read_csv(
    METADATA_PATH,
    dtype={column: "string" for column in metadata_text},
)
candles = pd.read_csv(
    CANDLES_PATH,
    dtype={"market_id": "string", "market_ticker": "string"},
)

for column in ["open_time", "close_time"]:
    metadata[column] = pd.to_datetime(metadata[column], utc=True, errors="coerce")
metadata["volume_fp"] = pd.to_numeric(metadata["volume_fp"], errors="coerce")
metadata["lifetime_days"] = pd.to_numeric(metadata["lifetime_days"], errors="coerce")
candles["date_utc"] = pd.to_datetime(candles["date_utc"], utc=True, errors="coerce")
for column in ["price_mean", "volume", "open_interest"]:
    candles[column] = pd.to_numeric(candles[column], errors="coerce")

data_inventory = pd.DataFrame(
    [
        {"file": "market metadata", "rows": len(metadata), "markets": metadata["market_id"].nunique()},
        {"file": "daily candles", "rows": len(candles), "markets": candles["market_id"].nunique()},
    ]
)
display(data_inventory)

## Define the research sample

We keep successful downloads with a resolved yes/no outcome and select the 50 highest-volume markets. This is a focused pilot sample, not a representative estimate for every Kalshi market.

In [ ]:
resolved = metadata.loc[
    metadata["download_status"].eq("success")
    & metadata["market_result"].isin(["yes", "no"])
].copy()
resolved["outcome"] = resolved["market_result"].map({"yes": 1, "no": 0})
top50 = (
    resolved.nlargest(50, "volume_fp")
    .sort_values("volume_fp", ascending=False)
    .reset_index(drop=True)
)
top50.insert(0, "volume_rank", range(1, len(top50) + 1))

sample_summary = pd.DataFrame(
    [
        {"metric": "resolved markets available", "value": len(resolved)},
        {"metric": "markets in pilot sample", "value": len(top50)},
        {"metric": "categories represented", "value": top50["series_category"].nunique()},
        {"metric": "sample yes-rate", "value": top50["outcome"].mean()},
    ]
)
display(sample_summary)

### Read the questions behind the numbers

The selected markets are shown with their questions, topics, outcomes, and volume so the sample remains interpretable rather than becoming an anonymous score table.

In [ ]:
display(
    top50[
        [
            "volume_rank", "market_id", "market_question",
            "series_category", "market_result", "volume_fp",
        ]
    ].head(10)
)

## Connect questions with daily probabilities

Only the daily rows belonging to the selected markets are brought into the research table. The final outcome stays beside each probability so the forecast can later be scored without a large all-market join.

In [ ]:
metadata_for_research = [
    "market_id", "market_question", "event_question",
    "series_title", "series_category", "market_result",
    "close_time", "outcome", "volume_fp",
]
research_candles = candles.loc[
    candles["market_id"].isin(top50["market_id"])
].merge(
    top50[metadata_for_research],
    on="market_id",
    how="inner",
    validate="many_to_one",
)

display(
    research_candles[
        [
            "market_id", "date_utc", "price_mean",
            "market_question", "close_time", "outcome",
        ]
    ].head(10)
)

## Set the forecast rule

We treat `price_mean` as the market's probability forecast. At each horizon, we use the latest daily value available no later than that many calendar days before close.

In [ ]:
forecast_horizons = [1, 3, 7]
protocol = pd.DataFrame(
    [
        {"forecast": "price_mean", "meaning": "daily market probability"},
        {"forecast": "0.5", "meaning": "uninformative reference forecast"},
        {"forecast": "sample yes-rate", "meaning": "ex-post frequency reference"},
    ]
)
display(protocol)

### Select one probability for each market and horizon

This is the only forecast-construction step: remove rows without a probability or outcome, apply the date cutoff, and keep the latest eligible daily observation for each market.

In [ ]:
forecast_rows = []

for horizon_days in forecast_horizons:
    eligible = research_candles.dropna(
        subset=["price_mean", "date_utc", "close_time", "outcome"]
    ).copy()
    cutoff = eligible["close_time"].dt.normalize() - pd.Timedelta(days=horizon_days)
    eligible = eligible.loc[eligible["date_utc"] <= cutoff]
    selected = (
        eligible.sort_values(["market_id", "date_utc"])
        .groupby("market_id", sort=False)
        .tail(1)
        .copy()
    )
    selected["horizon_days"] = horizon_days
    selected["forecast_date_utc"] = selected["date_utc"]
    selected["forecast_probability"] = selected["price_mean"].clip(0, 1)
    selected["actual_lead_days"] = (
        selected["close_time"].dt.normalize() - selected["forecast_date_utc"]
    ).dt.days
    forecast_rows.append(
        selected[
            [
                "market_id", "market_question", "series_category",
                "market_result", "outcome", "volume_fp",
                "horizon_days", "forecast_date_utc", "actual_lead_days",
                "forecast_probability",
            ]
        ]
    )

forecast_rows = pd.concat(forecast_rows, ignore_index=True)
forecast_coverage = (
    forecast_rows.groupby("horizon_days", as_index=False)
    .agg(
        markets=("market_id", "nunique"),
        median_actual_lead_days=("actual_lead_days", "median"),
        minimum_actual_lead_days=("actual_lead_days", "min"),
    )
)
display(forecast_coverage)

### Inspect the selected forecasts

These rows are the actual inputs to the scoring step. The lead-time column shows how close the selected observation was to the nominal horizon.

In [ ]:
display(
    forecast_rows[
        [
            "market_id", "market_question", "horizon_days",
            "forecast_date_utc", "actual_lead_days",
            "forecast_probability", "outcome",
        ]
    ].sort_values(["horizon_days", "market_id"]).head(10)
)

## Compare probabilistic accuracy

Brier Score is the mean squared difference between the market probability and the final outcome. Lower values are better; the table also shows the two reference forecasts.

In [ ]:
forecast_rows["squared_error"] = (
    forecast_rows["forecast_probability"] - forecast_rows["outcome"]
) ** 2
brier_summary = (
    forecast_rows.groupby("horizon_days", as_index=False)
    .agg(
        markets=("market_id", "nunique"),
        brier_score=("squared_error", "mean"),
        realized_yes_rate=("outcome", "mean"),
        median_actual_lead_days=("actual_lead_days", "median"),
    )
)
brier_summary["baseline_0_5"] = 0.25
brier_summary["sample_rate_reference"] = (
    brier_summary["realized_yes_rate"]
    * (1 - brier_summary["realized_yes_rate"])
)
brier_summary["gain_vs_0_5"] = (
    brier_summary["baseline_0_5"] - brier_summary["brier_score"]
)
display(brier_summary.round(4))

## Look at topic differences

The one-day result is summarized by category. These comparisons are exploratory because the top-50 sample is small and unevenly distributed across topics.

In [ ]:
one_day = forecast_rows.loc[forecast_rows["horizon_days"].eq(1)].copy()
category_summary = (
    one_day.groupby("series_category", dropna=False, as_index=False)
    .agg(
        markets=("market_id", "nunique"),
        brier_score=("squared_error", "mean"),
        realized_yes_rate=("outcome", "mean"),
    )
    .sort_values("markets", ascending=False)
)
category_summary["baseline_0_5"] = 0.25
category_summary["gain_vs_0_5"] = (
    category_summary["baseline_0_5"] - category_summary["brier_score"]
)
display(category_summary.head(10).round(4))

## Check calibration

If markets assign around 70%, those markets should resolve yes roughly 70% of the time. The table and chart below check that relationship for the one-day forecasts.

In [ ]:
calibration_bins = [-0.001, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.001]
calibration_labels = [
    "0-10%", "10-20%", "20-30%", "30-40%", "40-50%",
    "50-60%", "60-70%", "70-80%", "80-90%", "90-100%",
]
calibration_input = one_day.copy()
calibration_input["probability_bin"] = pd.cut(
    calibration_input["forecast_probability"],
    bins=calibration_bins,
    labels=calibration_labels,
    include_lowest=True,
)
calibration_summary = (
    calibration_input.groupby("probability_bin", observed=False, as_index=False)
    .agg(
        markets=("market_id", "nunique"),
        mean_forecast=("forecast_probability", "mean"),
        observed_yes_rate=("outcome", "mean"),
    )
)
display(calibration_summary.head(10).round(4))

### See calibration visually

Points near the diagonal have forecast probabilities close to the observed yes-rate. The labels show how many markets support each occupied probability range; empty ranges are omitted.

In [ ]:
plot_data = calibration_summary.loc[calibration_summary["markets"].gt(0)].dropna(
    subset=["mean_forecast", "observed_yes_rate"]
)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfect calibration")
ax.scatter(
    plot_data["mean_forecast"],
    plot_data["observed_yes_rate"],
    s=plot_data["markets"] * 8 + 20,
    alpha=0.8,
)
for _, row in plot_data.iterrows():
    ax.annotate(
        f"{row['probability_bin']}\n(n={int(row['markets'])})",
        (row["mean_forecast"], row["observed_yes_rate"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean forecast probability")
ax.set_ylabel("Observed yes-rate")
ax.set_title(f"One-day forecast calibration (n={len(one_day)})")
ax.legend()
plt.tight_layout()
plt.show()

## Inspect concrete successes and failures

The largest one-day errors make the aggregate score easier to discuss. Each row keeps the original question and the probability that produced the error.

In [ ]:
case_study = one_day[
    [
        "market_id", "market_question", "series_category",
        "market_result", "forecast_date_utc", "actual_lead_days",
        "forecast_probability", "outcome", "squared_error",
    ]
].sort_values("squared_error", ascending=False)
display(case_study.head(10).round(4))

## What this pilot can and cannot show

The main result is the Brier Score table, with calibration and category summaries providing context. A lower score than the 50% reference would support the narrow claim that these high-volume markets contain information about their resolved outcomes.

This is not yet a population-level result: the sample is selected by volume, only resolved markets are included, the sample-rate reference is calculated after observing outcomes, and the calendar lead time can be longer than the nominal horizon when daily observations are missing. The next research step is to expand beyond the top 50 and add uncertainty intervals.